[Reference](https://pub.towardsai.net/data-drift-in-production-ml-the-complete-detection-and-monitoring-guide-d570de8eb41e)

# Statistical Tests

## Kolmogorov-Smirnov (KS) Test

In [1]:
from scipy.stats import ks_2samp

# KS test between training and production feature
statistic, p_value = ks_2samp(training_feature, production_feature)

print(f"KS Statistic: {statistic:.4f}")  # Higher = more different
print(f"P-value: {p_value:.6f}")          # < 0.05 suggests drift

## Chi-Square Test (for categorical features)

In [2]:
from scipy.stats import chi2_contingency
import pandas as pd

# Create contingency table
contingency = pd.crosstab(
    training_data['categorical_feature'],
    production_data['categorical_feature']
)
chi2, p_value, dof, expected = chi2_contingency(contingency)
if p_value < 0.05:
    print("Categorical drift detected!")

# Distance Metrics
## Population Stability Index (PSI)

In [4]:
import numpy as np

def calculate_psi(expected, actual, bins=10):
    """
    Calculate Population Stability Index

    PSI = sum((actual% - expected%) * ln(actual% / expected%))

    Interpretation:
    PSI < 0.1: No significant population change
    0.1 ≤ PSI < 0.25: Small population change
    PSI ≥ 0.25: Large population change
    """

    # Bin the data
    expected_pct = pd.cut(expected, bins=bins).value_counts(normalize=True)
    actual_pct = pd.cut(actual, bins=bins).value_counts(normalize=True)

    # Calculate PSI
    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))

    return psi

# Usage
psi_value = calculate_psi(training_feature, production_feature)
if psi_value > 0.1:
    print(f"Data drift detected! PSI = {psi_value:.4f}")

## Kullback-Leibler (KL) Divergence


In [5]:
from scipy.spatial.distance import jensenshannon

# Calculate Jensen-Shannon divergence (symmetric KL)
# This is more stable than raw KL divergence
divergence = jensenshannon(training_dist, production_dist)

if divergence > 0.1:
    print(f"Significant divergence detected: {divergence:.4f}")

In [ ]:
# Decision rule
if p_value < 0.05:
    print("Drift is statistically significant. Action recommended.")
else:
    print("No significant drift detected. Continue monitoring.")

# Detection Methods: A Comprehensive Approach

## Method 1: Statistical Feature Monitoring

In [6]:
# Track feature statistics over time
feature_stats = {
    'mean': production_data['feature'].mean(),
    'median': production_data['feature'].median(),
    'std': production_data['feature'].std(),
    'q25': production_data['feature'].quantile(0.25),
    'q75': production_data['feature'].quantile(0.75),
    'min': production_data['feature'].min(),
    'max': production_data['feature'].max()
}

# Compare to training baseline
drift_indicators = {}
for stat_name, stat_value in feature_stats.items():
    baseline_value = training_stats[stat_name]
    pct_change = abs((stat_value - baseline_value) / baseline_value)

    if pct_change > 0.1:  # >10% change is significant
        drift_indicators[stat_name] = pct_change
if drift_indicators:
    print(f"Potential drift in: {drift_indicators}")


## Method 2: Unsupervised Drift Detection (When Labels Unavailable)

### Autoencoder Reconstruction Error

In [7]:
# Pseudocode
autoencoder = train_autoencoder(training_data)

# Monitor reconstruction error on production data
reconstruction_errors = []
for batch in production_data:
    reconstructed = autoencoder.predict(batch)
    error = mean_squared_error(batch, reconstructed)
    reconstruction_errors.append(error)

# If mean reconstruction error increases significantly, drift detected
if np.mean(reconstruction_errors) > threshold:
    print("Data drift detected via reconstruction error!")

## Method 3: Model Performance Monitoring (When Labels Available)


In [8]:
# Track performance over time
performance_history = {
    'week_1_accuracy': 0.94,
    'week_2_accuracy': 0.91,
    'week_3_accuracy': 0.87,
    'week_4_accuracy': 0.82
}

# Calculate degradation
baseline_accuracy = performance_history['week_1_accuracy']
current_accuracy = performance_history['week_4_accuracy']
degradation = baseline_accuracy - current_accuracy

if degradation > 0.05:  # >5% drop is significant
    print(f"Performance degradation detected: {degradation:.2%}")

# What to Monitor: The Essential Metrics
- Feature statistics: Mean, median, std, min, max for each numeric feature
- PSI or KL divergence: For each feature
- Model performance: Accuracy, precision, recall, F1 on labelled data
- Prediction distribution: How predictions change over time
- Data quality: Missing values, outliers, data types

## Retraining Triggers

In [9]:
MONITORING_RULES = {
    'psi_threshold': 0.15,           # PSI > 0.15 = retrain
    'accuracy_drop': 0.05,            # 5% accuracy drop = retrain
    'ks_statistic_threshold': 0.1,   # KS > 0.1 = investigate
    'manual_trigger': True             # Business teams can trigger
}

# Check drift daily
if calculated_psi > MONITORING_RULES['psi_threshold']:
    trigger_retraining()

## Tools for Production Drift Detection (2025)

In [10]:
# Using Evidently AI (popular open-source library)
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

report = Report(metrics=[DataDriftPreset()])
report.run(reference_data=training_data, current_data=production_data)
report.save_html('drift_report.html')  # Visual drift report